### Resolving $k$ Odd When $n$ is Even

In Section 5 of the manuscript, the cases where $k$ is odd ($k \in \{5, 7, 9, 11\}$) and $n$ is even are examined. By dividing out the polynomial factors of $S_k(x)$, the problem reduces to checking two distinct conditions depending on whether the prime $p$ divides $T_k(x)$. 

This script executes the associated computational checks for these cases:
* **Elliptic Curve Searches:** When $p \nmid T_k(x)$ for $k \in \{5, 7, 9\}$, the equations transform into distinct families of elliptic curves. The script computes the integral points on these curves, reverses the changes of variables, and filters for valid positive integer candidates for $x$.
* **Binomial Thue Equations:** When $p \mid T_k(x)$ for $k \in \{7, 11\}$, the problem reduces to solving a finite set of binomial Thue equations of the form $Aa^4 - Bb^4 = \pm 1$. These are solved unconditionally utilizing PARI/GP, filtering out trivial solutions.
* **Pell Sequence Solutions ($k=7$):** As noted in Section 5.2.1, the case $k=7$ yields sparse solutions via the Pell sequence $P_{4m+2}$. The script verifies a pair of solutions, calculating the 79-digit prime resulting from the $m=13$ index.

In [ ]:
# =============================================================================
# Computations for k = 5, 7, 9, 11 (n even)
# =============================================================================

R.<x, y> = PolynomialRing(QQ)

def S(k):
    """Returns S_k(x) as a polynomial in x using Faulhaber's formula."""
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def solve_thue_unconditional(f, m):
    """Unconditionally solves the Thue equation f = m without GRH."""
    assert f.is_homogeneous()
    parithueinit = gp.thueinit(f.subs({f.variables()[1]: 1}), flag=1)
    return gp.thue(parithueinit, m).sage()

def check_odd_k_elliptic_curves():
    """
    Computes integral points for the elliptic curves arising from 
    k = 5, 7, 9 when p does not divide T_k(x). Filters for valid 
    positive integer candidate x-values.
    """
    print("=" * 75)
    print("Elliptic Curve Searches (p ∤ T_k(x))")
    print("=" * 75)
    
    # --- k = 5 ---
    print("\nChecking k=5: Y^2 = X^3 + 6CX")
    for C in [1, 3]:
        EC = EllipticCurve([0, 0, 0, 6 * C, 0])
        pts = EC.integral_points()
        found_valid = False
        for pt in pts:
            x_val = pt[0] / (2 * C)
            w_val = pt[1] / (2 * C)
            if x_val in ZZ and x_val > 0:
                print(f"  [VALID] C={C:<2} | pt={str(pt):<15} | x={str(x_val):<5} | w={w_val}")
                found_valid = True
        if not found_valid:
            print(f"  C={C:<2} | No positive integer x candidates found.")
            
    # --- k = 7 ---
    print("\nChecking k=7: Y^2 = X^3 + (792/C)X^2 + (198288/C^2)X")
    for C in [2, 6]:
        EC_7 = EllipticCurve([0, 132 * (6 / C), 0, (6 / C)^2 * 5508, 0])
        pts = EC_7.integral_points()
        found_valid = False
        for pt in pts:
            x_val = pt[0] / 18
            w_val = pt[1] / 48
            if x_val in ZZ and x_val > 0:
                print(f"  [VALID] C={C:<2} | pt={str(pt):<15} | x={str(x_val):<5} | w={w_val}")
                found_valid = True
        if not found_valid:
            print(f"  C={C:<2} | No positive integer x candidates found.")
            
    # --- k = 9 ---
    print("\nChecking k=9: Y^2 = X^3 - 5CX^2 + 12C^2X - 12C^3")
    for C in [1, 5]:
        EC_9 = EllipticCurve([0, -5 * C, 0, C^2 * 12, - C^3 * 12])
        pts = EC_9.integral_points()
        found_valid = False
        for pt in pts:
            x_val = pt[0] / (2 * C)
            w_val = pt[1] / (2 * C^2)
            if x_val in ZZ and x_val > 0:
                print(f"  [VALID] C={C:<2} | pt={str(pt):<15} | x={str(x_val):<5} | w={w_val}")
                found_valid = True
        if not found_valid:
            print(f"  C={C:<2} | No positive integer x candidates found.")

def check_k7_k11_thue_equations():
    """
    Computes unconditional solutions to the binomial Thue equations 
    arising from k = 7, 11 when p divides T_k(x). Filters out 
    trivial solutions (e.g., x=0, y=0, or +/- 1).
    """
    print("\n" + "=" * 75)
    print("Binomial Thue Equations for k=7, 11 (p | T_k(x))")
    print("=" * 75)
    
    equations_to_check = set()
    
    for r in [1, 2, 3]:
        for m in [-1, 1]:
            equations_to_check.add((x^4 - 2^r * y^4, m))
            
    for p in [1, 3, 5]:
        for r in [1, 2, 3]:
            for m in [-1, 1]:
                equations_to_check.add((p^r * x^4 - 2 * y^4, m))
                equations_to_check.add((x^4 - 2 * p^r * y^4, m))
                
    print(f"{'Equation':<20} | {'m':<4} | {'Solutions (X, Y)'}")
    print("-" * 75)
    
    for eq, m in sorted(list(equations_to_check), key=lambda item: str(item[0])):
        sols = solve_thue_unconditional(eq, m)
        
        # Filter for non-trivial solutions: ignore x=0, y=0, or +/- 1
        non_trivial = [sol for sol in sols if abs(sol[0]) > 1 or abs(sol[1]) > 1]
        
        if non_trivial:
            print(f"{str(eq):<20} | {str(m):<4} | {str(non_trivial)}")
        else:
            print(f"{str(eq):<20} | {str(m):<4} | Trivial solutions only")

def check_k5_f_curves():
    """
    Computes integral points for the elliptic curves arising from 
    k = 5 when p divides T_5(x). 
    Curves: F_{5, c, ±}: Y^2 = X^3 ∓ 4cX^2 + 6c^2X
    """
    print("\n" + "=" * 75)
    print("Checking k=5 F-curves (p | T_5(x))")
    print("=" * 75)

    for c in [1, 3]:
        # Test both the minus (-) and plus (+) cases for the X^2 coefficient
        for sign_val, sign_str in [(-1, '+'), (1, '-')]:
            # If sign_str is '-', coefficient is -4c. If '+', coefficient is +4c.
            b_coef = -4 * c if sign_str == '-' else 4 * c
            
            EC = EllipticCurve([0, b_coef, 0, 6 * c^2, 0])
            pts = EC.integral_points()
            found_valid = False
            
            for pt in pts:
                X, Y = pt[0], pt[1]
                
                # Check substitution constraints: X = 2 * c^2 * r^4
                r4 = X / (2 * c^2)
                if r4 > 0 and r4.is_integer():
                    try:
                        # Exact root extraction in ZZ; throws ValueError if not a perfect 4th power
                        r = ZZ(r4).nth_root(4)
                        
                        if r > 0:
                            # Y = 2 * c^2 * r^2 * (2x + 1)
                            if Y % (2 * c^2 * r^2) == 0:
                                v = Y / (2 * c^2 * r^2)
                                x_val = (v - 1) / 2
                                
                                if x_val in ZZ and x_val > 0:
                                    print(f"  [VALID] c={c:<2}, sign={sign_str} | pt={str(pt):<15} | x={x_val}")
                                    found_valid = True
                    
                    except ValueError:
                        # r4 is not a perfect 4th power; skip this point
                        continue
                                
            if not found_valid:
                print(f"  c={c:<2}, sign={sign_str} | No positive integer x candidates found.")

def check_k9_f_curves():
    """
    Computes integral points for the elliptic curves arising from 
    k = 9 when p divides x^2+x-1.
    Mapped from quartic to: Y^2 = X^3 - 16cX^2 + 124c^2X
    """
    print("\n" + "=" * 75)
    print("Checking k=9 F-curves (p | x^2+x-1)")
    print("=" * 75)

    for c in [1, 5]:
        EC = EllipticCurve([0, -16 * c, 0, 124 * c^2, 0])
        pts = EC.integral_points()
        found_valid = False
        
        for pt in pts:
            X, Y = pt[0], pt[1]
            
            # Check substitution constraints: X = 2 * c * v^2  (where v = 2x + 1)
            v2 = X / (2 * c)
            if v2 > 0 and v2.is_integer():
                v = sqrt(v2)
                if v in ZZ:
                    x_val = (v - 1) / 2
                    
                    if x_val in ZZ and x_val > 0:
                        print(f"  [VALID] c={c:<2} | pt={str(pt):<15} | x={x_val}")
                        found_valid = True
                        
        if not found_valid:
            print(f"  c={c:<2} | No positive integer x candidates found.")

def verify_k7_pell_solutions():
    """
    Verifies the massive solutions to S_7(x) = p * y^4 discovered via 
    Pell sequences, corresponding to Section 5.2.1 of the manuscript.
    """
    print("\n" + "=" * 75)
    print("Verifying Pell Sequence Solutions for k=7, n=4")
    print("=" * 75)
    
    def Pell(n):
        if n == 0: return 0
        if n == 1: return 1
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, 2 * b + a
        return b

    for m_val in [1, 13]:
        idx = 4 * m_val + 2
        P_idx = Pell(idx)
        
        w = P_idx // 2  
        
        # Recover x from x(x+1) = 2w^2 => x^2 + x - 2w^2 = 0
        x_val = (-1 + isqrt(1 + 8 * w^2)) // 2
        
        print(f"Testing m = {m_val} (Pell Index P_{idx})")
        print(f"  -> w = {w}")
        print(f"  -> x = {x_val}")
        
        # Calculate S_7(x)
        s_val = ZZ(S(7)(x=x_val))
        
        # We expect S_7(x) = p * y^4, where y = w. Find p.
        y_val = w
        
        p_candidate = s_val // (y_val^4)  

        # Since p_candidate is now in ZZ, .is_prime() will evaluate correctly
        if p_candidate.is_prime():
            p_str = str(p_candidate)
            # Truncate the 79-digit prime for clean console output
            if len(p_str) > 20:
                p_str = p_str[:10] + f"... [{len(p_str)} digits total] ..." + p_str[-10:]
                
            print(f"  [VALID] S_7({x_val}) = {p_str} * {y_val}^4")
        else:
            print(f"  [FAILED] Remainder is not prime: {p_candidate}")
        print("-" * 75)

# =============================================================================
# Execution Pipeline
# =============================================================================

check_odd_k_elliptic_curves()
check_k5_f_curves()             # <-- Added
check_k9_f_curves()             # <-- Added
check_k7_k11_thue_equations()
verify_k7_pell_solutions()

Elliptic Curve Searches (p ∤ T_k(x))

Checking k=5: Y^2 = X^3 + 6CX
  C=1  | No positive integer x candidates found.
  [VALID] C=3  | pt=(6 : -18 : 1)   | x=1     | w=-3
  [VALID] C=3  | pt=(72 : -612 : 1) | x=12    | w=-102

Checking k=7: Y^2 = X^3 + (792/C)X^2 + (198288/C^2)X
  [VALID] C=2  | pt=(612 : -20196 : 1) | x=34    | w=-1683/4
  C=6  | No positive integer x candidates found.

Checking k=9: Y^2 = X^3 - 5CX^2 + 12C^2X - 12C^3
  [VALID] C=1  | pt=(2 : 0 : 1)     | x=1     | w=0
  [VALID] C=5  | pt=(10 : 0 : 1)    | x=1     | w=0
  [VALID] C=5  | pt=(20 : -50 : 1)  | x=2     | w=-1

Checking k=5 F-curves (p | T_5(x))
  c=1 , sign=+ | No positive integer x candidates found.
  c=1 , sign=- | No positive integer x candidates found.
  c=3 , sign=+ | No positive integer x candidates found.
  c=3 , sign=- | No positive integer x candidates found.

Checking k=9 F-curves (p | x^2+x-1)
  c=1  | No positive integer x candidates found.
  [VALID] c=5  | pt=(90 : -600 : 1) | x=1

Binomial Th